# 2.6 Pipelines

El pipeline integra el preprocesamiento y el modelo en un solo objeto. Así, la API podrá recibir datos originales —por ejemplo, edad `46` y sexo `male`— y aplicar automáticamente las mismas transformaciones usadas durante el entrenamiento.

## Importar paquetes

In [1]:
import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score

## Carga de datos

In [2]:
df = pd.read_csv('./data/titanic_clean.csv')
print(f'Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas')
print(f'Valores nulos: {df.isna().sum().sum()}')
df.head()

Dimensiones: 891 filas y 8 columnas
Valores nulos: 0


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## Preprocesamiento

`ColumnTransformer` aplica One-Hot Encoding a las variables categóricas y `QuantileTransformer` a `Age` y `Fare`. Las columnas restantes pasan sin modificación. One-Hot Encoding crea columnas binarias y evita imponer un orden artificial a categorías como los puertos; por eso no es equivalente técnicamente a `LabelEncoder`.

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'), ['Sex', 'Embarked']),
        ('age', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Age']),
        ('fare', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Fare'])
    ],
    remainder='passthrough'
)

## Definición de modelos

El prefijo `model__` indica que el hiperparámetro pertenece al paso llamado `model` dentro del pipeline.

In [4]:
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'model__C': [0.01, 0.1, 1, 10, 100],
            'model__penalty': ['l1', 'l2'],
            'model__solver': ['liblinear', 'saga'],
            'model__max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(random_state=42),
        'parametros': {
            'model__splitter': ['best', 'random'],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(random_state=42),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(random_state=42),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(random_state=42),
        'parametros': {'model__n_estimators': [10, 100]}
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {'model__n_neighbors': [3, 5, 7]}
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(random_state=42),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(random_state=42),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3],
            'model__learning_rate': [0.1, 0.2, 0.3],
            'model__verbose': [-1]
        }
    },
    'GaussianNB': {'modelo': GaussianNB(), 'parametros': {}},
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {'model__alpha': [0.1, 1.0, 10.0]}
    }
}

## División de datos

In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=100, stratify=y
)

print(f'Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}')

Entrenamiento: (712, 7) | Prueba: (179, 7)


## Entrenamiento con Pipeline y GridSearchCV

In [6]:
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

for nombre, info_modelo in modelos.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', MinMaxScaler()),
        ('model', info_modelo['modelo'])
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    y_pred = grid_search.predict(X_test)
    precision = accuracy_score(y_test, y_pred)

    puntajes_modelos.append({'Modelo': nombre, 'Precisión': precision})
    estimadores[nombre] = grid_search.best_estimator_

    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)
print('Rendimiento de los modelos de clasificación')
print(metricas.round(2).to_string(index=False))
print('---------------------------------------------------')
print('MEJOR MODELO DE CLASIFICACIÓN')
print(f'Modelo: {mejor_modelo}')
print(f'Precisión: {mejor_precision:.2f}')

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To 

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf inst

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To 

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf inst

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty'

...

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuel

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuel

/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuela/aprendizajesupervisado/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/disenoux/Documents/escuel

Rendimiento de los modelos de clasificación
                             Modelo  Precisión
               Clasificador XGBoost       0.85
 Clasificador de Bosques Aleatorios       0.85
  Clasificador de Gradient Boosting       0.84
Clasificador de Vectores de Soporte       0.84
                  Clasificador LGBM       0.84
                Regresión Logística       0.83
  Clasificador de Árbol de Decisión       0.83
           Clasificador Naive Bayes       0.81
              Clasificador AdaBoost       0.79
                         GaussianNB       0.79
   Clasificador K-Nearest Neighbors       0.79
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador XGBoost
Precisión: 0.85


## Guardar y validar el pipeline

In [7]:
with open('pipeline.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

nuevo_pasajero = pd.DataFrame([{
    'Pclass': 2,
    'Sex': 'male',
    'Age': 46,
    'SibSp': 0,
    'Parch': 0,
    'Fare': 7.25,
    'Embarked': 'C'
}])

with open('pipeline.pkl', 'rb') as archivo_estimador:
    pipeline_cargado = pickle.load(archivo_estimador)

prediccion = pipeline_cargado.predict(nuevo_pasajero)
print('Pipeline guardado en pipeline.pkl')
print(f'Predicción de prueba: {prediccion}')

Pipeline guardado en pipeline.pkl
Predicción de prueba: [0]


> `pipeline.pkl` contiene tanto las transformaciones ajustadas como el modelo ganador. Debe cargarse únicamente desde una fuente confiable y con versiones compatibles de las dependencias.